In [1]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
print("pasta data criada")

pasta data criada


In [2]:
import shutil
from pathlib import Path

for pdf in Path(".").glob("*.pdf"):
    shutil.move(str(pdf), f"data/{pdf.name}")

print("PDFs movidos para data/")

PDFs movidos para data/


In [3]:
for f in Path("data").glob("*.pdf"):
    print(f)

data/amoxicilina.pdf
data/omeprazol.pdf
data/ibuprofeno.pdf
data/paracetamol.pdf
data/dipirona.pdf


In [4]:
!pip install -U pypdf chromadb langchain langchain-community langchain-text-splitters sentence-transformers

In [5]:
from pypdf import PdfReader
from pathlib import Path

texts = []

for pdf in Path("data").glob("*.pdf"):

    reader = PdfReader(pdf)

    texto = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            texto += page_text

    texts.append({
        "arquivo": pdf.name,
        "texto": texto
    })

print("PDFs carregados:", len(texts))

# teste rápido
print(texts[0]["arquivo"])
print(texts[0]["texto"][:500])

PDFs carregados: 5
amoxicilina.pdf
AAS 
amoxicilina 
Aché Laboratórios Farmacêuticos S.A. 
Pó para suspensão 
250 mg/5ml 
 
amoxicilina 250 pó sus _BU09_VP_GO 
BULA PARA PACIENTE 
Bula de acordo com a Resolução-RDC nº 47/2009 
 
I – IDENTIFICAÇÃO DO MEDICAMENTO 
 
amoxicilina  
Medicamento Genérico, Lei nº 9.787, de 1999 
 
APRESENTAÇÃO  
Pó para suspensão  de 250 mg/5 ml: embalagem contendo 1 frasco com pó para suspensão + 1 seringa 
dosadora. 
 
USO ORAL  
USO ADULTO E PEDIÁTRICO 
 
COMPOSIÇÃO 
Cada 5 ml de suspensão contém: 
a


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " "]
)
chunks = []

for item in texts:
    partes = splitter.split_text(item["texto"])

    for p in partes:
        chunks.append({
            "texto": p,
            "arquivo": item["arquivo"]
        })

print("Chunks criados:", len(chunks))

Chunks criados: 181


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_texts(
    texts=[c["texto"] for c in chunks],
    embedding=embeddings
)

/tmp/ipykernel_42602/3495461776.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [9]:
def buscar_resposta(pergunta):

    resultados = db.similarity_search(pergunta, k=4)

    contexto = "\n\n".join([r.page_content for r in resultados])

    return contexto

In [10]:
print(buscar_resposta("para que serve ibuprofeno"))

1 Bula_Paciente 
ibuprofeno 
Vitamedic Indúst
ria Farmacêutica Ltda 
Suspensão Gotas 
100mg/mL 2 Bula_Paciente 
ibuprofeno 
Medicamento genérico, Lei nº 9.787, de 1999 
I – IDENTIFICAÇÃO DO MEDICAMENTO: 
APRESENTAÇÕES 
Suspensão Gotas.  
Embalagens com: 1, 50, 100 ou 200 frascos com 20mL. 
USO ORAL  
USO ADULTO E PEDIÁTRICO ACIMA DE 6 MESES  
COMPOSIÇÃO 
Cada mL da suspensão gotas* contém: 
ibuprofeno......................................................................................................................................................................100mg 
veículo q.s.p.......................................................................................................................................................................1mL

Excipientes: goma xantana, glicerol, benzoato de sódio, propilenoglicol, ácido cítrico, aroma tutti-frutti, sorbitol, sacarina 
sódica dihidratada, ciclamato de sódio, dióxido de titânio, po lissorbato 80, essência de baunilha hidrossolúv